# Target Tracking — Models, Polar Conversion, IMM & α-β-γ Filters

*Course 2 — Linear Kalman Filter, Part 5. The classic KF application: tracking a moving target whose driving input is unknown. Uses the kinematic models from [02](02_KinematicModels_NCV_NCA_NCT_Implementation.ipynb) and everything from [08](08_Deriving_the_Linear_Kalman_Filter.ipynb)–[10](10_KF_Extensions_FaultDetection_SteadyState_Smoothing.ipynb).*

**Style:** every equation gets a plain-language paraphrase (→); extra intuition is flagged **→ Intuition**.

### 🧩 What Makes Target Tracking Special

- We want to localize a target (friendly or hostile), but we **do not know its control input** $u_k$ — the target chooses its own maneuvers.

  → Since $u_k$ is unknown and arguably non-Gaussian, we lean on simple **physics-based motion models** and treat maneuvers as process noise.

- We use the kinematic models from [02](02_KinematicModels_NCV_NCA_NCT_Implementation.ipynb): **NCP** (nearly-constant position), **NCV** (nearly-constant velocity), **NCA** (nearly-constant acceleration), **CT** (coordinated turn).

- **→ Intuition:** even when position/velocity estimates are imperfect, the **covariance** still has a useful geometric meaning — a hyper-ellipsoid pinpointing *where the target probably is*, which tells a sensor **where to look next**. The uncertainty is itself an actionable output.

### 🧩 Discrete Tracking Models

- **2-D NCV** (state $[\xi,\dot\xi,\eta,\dot\eta]^T$, sample $\Delta t$):

$$
x_{k+1} = \begin{bmatrix} 1 & \Delta t & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 1 & \Delta t \\ 0 & 0 & 0 & 1\end{bmatrix}x_k + w_k, \qquad z_k = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 0 & 1 & 0\end{bmatrix}x_k + v_k.
$$

  → Position integrates velocity each step; the process noise $w_k$ stands in for the unknown small accelerations (maneuvers). We measure position only.

- **2-D coordinated turn (CT)** with turn rate $\Omega$ mixes the two velocity components through $\sin(\Omega\Delta t),\cos(\Omega\Delta t)$ terms — a constant-speed circular motion model.

  → **NCP/NCV/NCA/CT** form a small library of maneuver hypotheses; picking the right one (or blending them — see IMM below) is most of the tracking art.

- **→ Intuition:** the tracker is often only *one component*; a full system also needs to **detect** targets in sensor frames, **associate** measurements to tracks, and manage track lifecycles (next).

### 🧩 Gating, Association, and Track Lifecycle

- **Detection:** subtract a local background (median of neighbors) from each pixel, threshold, and call the survivors **exceedances** $E_{i,j}$ — candidate target pixels.

- **Gating & association:** put a rectangular **track gate** of size $G$ pixels around each track's predicted position $\hat\xi_k^- = A\hat\xi_{k-1}^+$; exceedances inside it belong to that track. The measurement is their intensity-weighted **centroid**:

$$
z_k = \frac{\sum_{m,n\in\text{Gate}} m\,E_{m,n}}{\sum_{m,n\in\text{Gate}} E_{m,n}}.
$$

  → Predict where the target should be, draw a box there, and average the bright pixels inside to get one clean position measurement. Gate **loose** when predictions are unreliable, **tight** when they agree with data (less noise let in).

- **Track lifecycle:** *start* a track only after a candidate persists several frames (reject glints); *coast* (open gate, run open-loop) when a target is briefly obscured; *merge* crossing tracks; *delete* a track after too many missed detections.

- **→ Intuition:** the KF answers "where is this one target?"; the surrounding logic answers "how many targets are there and which measurement is which?" — the harder real-world problem.

### 🧩 Polar Measurements with a Cartesian State

- Radar/sonar return **polar** measurements (range $r$, bearing $\theta$) but the dynamics are cleanest in **Cartesian** coordinates. Two options: (1) a nonlinear KF with a polar output equation, or (2) convert the measurement to Cartesian and use a linear KF.

- The **naïve** conversion $x_m = r_m\cos\theta_m,\ y_m = r_m\sin\theta_m$ is **biased**, because the nonlinear map distorts the noise. The mean error is

$$
\bar{v}(r,\theta) = \begin{bmatrix} r\cos\theta\,(e^{-\sigma_\theta^2/2}-1) \\ r\sin\theta\,(e^{-\sigma_\theta^2/2}-1)\end{bmatrix}.
$$

  → Converting noisy angles through cosine/sine **shrinks the average radius** — a systematic bias that grows with range and bearing noise. A KF is very sensitive to bias, so this must be corrected.

- **Debiased consistent conversion:** compensate for the average bias to get an unbiased synthetic Cartesian measurement:

$$
z = \begin{bmatrix} r_m\cos\theta_m\,\big(1 - e^{-\sigma_\theta^2} + e^{-\sigma_\theta^2/2}\big) \\ r_m\sin\theta_m\,\big(1 - e^{-\sigma_\theta^2} + e^{-\sigma_\theta^2/2}\big)\end{bmatrix},
$$

  with a matching **converted-measurement covariance** $\Sigma_{\tilde v}$ (range/bearing-dependent trigonometric expressions, recomputed every step).

- **→ Intuition:** ironically, a nonlinear KF (Course 3) can be *less* work than getting these debiasing corrections right — but the debiased linear approach keeps the estimator linear and Gaussian-consistent.

### 🧩 The Interacting Multiple-Model (IMM) Filter

- A target may **switch modes** — cruise (NCV), then hover (NCP), then turn (CT). No single model fits. The **IMM** runs $M$ Kalman filters in parallel (one per mode) and intelligently blends them.

- It maintains a mode probability mass function $\mu_k$ (belief the target is in each mode) and a **mode-transition matrix**:

$$
p_{ij} = \Pr(m_k = j \mid m_{k-1} = i).
$$

  → $p_{ij}$ encodes how likely the target is to switch from mode $i$ to mode $j$ between steps — the state machine of maneuvers.

- The IMM output is a **blended** state, covariance, and updated mode-pmf:

$$
\hat{x}_k^{+} = \sum_{j=1}^{M}\hat{x}_{j,k}^{+}\mu_{j,k}, \qquad \Sigma_{\tilde{x},k}^{+} = \sum_{j=1}^{M}\Big\{\Sigma_{\tilde{x}j,k}^{+} + [\hat{x}_{j,k}^{+}-\hat{x}_k^{+}][\hat{x}_{j,k}^{+}-\hat{x}_k^{+}]^{T}\Big\}\mu_{j,k}.
$$

  → The combined estimate is the mode-probability-weighted average of the individual filters; the combined covariance adds a **"spread of the means"** term that inflates uncertainty when the filters *disagree* (mode ambiguity).

### 🧩 The Three IMM Steps

Run once per measurement interval:

- **1 · Interaction (mixing):** blend the prior filter outputs into each filter's new starting point, using mixing probabilities $\mu_{i|j,k-1} = \tfrac{1}{\bar c_j}p_{ij}\mu_{i,k-1}$:

$$
\hat{x}_{j,k-1}^{(\text{mod})} = \sum_{i=1}^{M}\hat{x}_{i,k-1}^{+}\,\mu_{i|j,k-1},
$$

  with a matching mixed covariance (prior covariances + spread-of-means).

  → Before each filter runs, seed it with a mixture of *all* filters' previous estimates, weighted by how likely mode $i$ led to mode $j$. This coupling is what makes the models "interacting."

- **2 · Filtering:** run each of the $M$ Kalman filters one step; also compute each mode's **measurement likelihood**

$$
\Lambda_{j,k} = \mathcal{N}\big(z_k - \hat{z}_{j,k},\, \Sigma_{\tilde{z}j,k}\big).
$$

  → How well did mode $j$ predict the actual measurement? High likelihood ⇒ that mode gains probability.

- **3 · Combination:** update the mode pmf $\mu_{j,k}\propto\Lambda_{j,k}\bar c_j$ and blend outputs (formulas above).

- **→ Intuition:** the IMM is a soft, probabilistic model-selector — it never commits to one model but continuously re-weights them by evidence. Excellent state *and* mode estimates, at $M\times$ the cost of one KF.

### 🧩 Steady-State α-β and α-β-γ Filters

- For the common **NCV** model, the steady-state KF ([10](10_KF_Extensions_FaultDetection_SteadyState_Smoothing.ipynb)) has a **closed-form** solution — the classic **α-β filter**:

$$
\hat{p}_{k+1} = \hat{p}_k + (\Delta t)\hat{v}_k + \alpha\,(z_k - \hat{p}_k), \qquad
\hat{v}_{k+1} = \hat{v}_k + \tfrac{\beta}{\Delta t}(z_k - \hat{p}_k).
$$

  → Predict position from velocity, then correct **position** by $\alpha\times$ residual and **velocity** by $\tfrac{\beta}{\Delta t}\times$ residual. Two constant gains, no online covariance math.

- The optimal $\alpha,\beta$ come from the **target-tracking (maneuvering) index**

$$
\lambda = \frac{\sigma_{\tilde{w}}(\Delta t)^2}{\sqrt{\Sigma_{\tilde{v}}}},
$$

  a single knob = (motion uncertainty)/(measurement uncertainty). The **NCA** model gives the analogous three-gain **α-β-γ filter** (adds acceleration correction $\gamma$).

  → Large $\lambda$ (agile target, clean sensor) ⇒ big gains, trust measurements; small $\lambda$ (steady target, noisy sensor) ⇒ small gains, trust the model. These match `dlqe` exactly for their models but need only **algebra**, no Riccati solver.

- **→ Intuition:** the venerable α-β / α-β-γ trackers used in radar for decades are just **steady-state Kalman filters in disguise** — this derivation shows *why* their magic constants are optimal.

### 🧩 Summary

- Target tracking uses simple maneuver models (**NCP/NCV/NCA/CT**) because the target's input is unknown; the covariance ellipsoid tells sensors where to look.

- A full tracker adds **detection, gating/association**, and **track lifecycle** management around the KF.

- **Polar → Cartesian** conversion must be **debiased** (the naïve conversion shrinks radius), producing a consistent synthetic measurement and covariance.

- The **IMM** runs $M$ mode-specific KFs, blending them through **interaction → filtering → combination** with a mode-transition matrix $p_{ij}$ — soft model selection.

- **α-β** (NCV) and **α-β-γ** (NCA) filters are closed-form steady-state KFs whose gains follow from the single **maneuvering index** $\lambda$.

---
*Course 2 complete. Next: [12 · The Extended Kalman Filter (EKF)](12_Extended_Kalman_Filter.ipynb) — nonlinear models.*